In [1]:
import torch
import time
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.swarmset import ContinuingDataset, SwarmDataset
from src.networks.embedding import NoveltyEmbedding
from src.networks.archive import DataAggregationArchive
from src.networks.ensemble import Ensemble
import numpy as np
from scipy import ndimage
import random
from sklearn.manifold import TSNE
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

def CSVLineToVec(line):
    line_list = line.strip().replace("\n", "").split(",")
    float_list = []
    for i in line_list:
        float_list.append(float(i))
    float_list = np.array(float_list)
    return float_list

pygame 2.1.2 (SDL 2.0.18, Python 3.10.14)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [5]:
# For single Sensor Baseline Model
# TRUTH_FILE = "validation-data-two-sensor.txt"
# TRUTH_FILE = "validation-data-baseline.txt"

"""
RANDOM - Only Random Weight Initialization
"""
# VALIDATION_FILE = "single.csv"
# VALIDATION_DATA = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/single", rank=0)
# TESTING_FILE = "single.csv"
# TESTING_DATA = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/single", rank=0)
# ENSEMBLE_PATH = None


"""
NETWORKED -- Learned Representation
"""
VALIDATION_FILE = "single.csv"
VALIDATION_DATA = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/single", rank=0)
TESTING_FILE = "single.csv"
TESTING_DATA = SwarmDataset("~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/single", rank=0)
ENSEMBLE_PATH = "~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/checkpoints/ensembles/patient0"

OUT = "~/Desktop/DevProjects/SwarmBehaviorDiscovery/src/z-experiments/Sum24-notebooks/data/oracle_labels"
validation_classes = []
with open(os.path.join(OUT, VALIDATION_FILE), "r") as f:
    lines = f.readlines()
    validation_classes = [-1 for i in range(len(lines))]
    for line in lines:
        triplet = CSVLineToVec(line)
        validation_classes[int(triplet[0])] = int(triplet[1])

testing_classes = []
with open(os.path.join(OUT, TESTING_FILE), "r") as f:
    lines = f.readlines()
    testing_classes = [-1 for i in range(len(lines))]
    for line in lines:
        triplet = CSVLineToVec(line)
        testing_classes[int(triplet[0])] = int(triplet[1])

validation_set = []
testing_set = []
for i, _class in enumerate(testing_classes):
    testing_set.append((i, _class))

for i, _class in enumerate(validation_classes):
    validation_set.append((i, _class))

print(len(validation_set), len(testing_set))

500 500


In [6]:
device = "cpu"
ensemble = Ensemble(size=1, output_size=5, lr=3e-2, weight_decay=1e-6, new_model=False, manual_schedulers=False, dynamic_lr=False, init="Random", warmup=5, margin=1)
if ENSEMBLE_PATH:
    ensemble.load_ensemble(ENSEMBLE_PATH, full=True)
ensemble.eval_mode()

MARGIN = 0
# for metric in ["TRAIN", "VAL"]:
for metric in ["VAL"]:
    a = []
    sampled_dataset = TESTING_DATA if metric == "TRAIN" else VALIDATION_DATA
    c_set = testing_set if metric == "TRAIN" else validation_set
    classes = testing_classes if metric == "TRAIN" else validation_classes

    # random.shuffle(c_set)
    print("=" * 20)
    print(f"{metric} results")
    print(f"Class Set of Size: {len(c_set)}")
    for i in range(len(ensemble.ensemble)):
        embedded_positions = []
        for j, c in enumerate(classes):
            image, _ = sampled_dataset[j][0], sampled_dataset[j][1][0]
            image = np.expand_dims(image, axis=0)
            embed = ensemble.ensemble[0].numpy_single_pass(image)
            embed = embed.detach().cpu().squeeze(dim=0).numpy()
            embedded_positions.append(embed)

        # Evaluate Accuracy
        correct, total = 0, 0
        class_counts = {i:0 for i in range(max(classes) + 1)}
        class_accuracy = {i:0 for i in range(max(classes) + 1)}
        for l in range(len(c_set)):
            x, _classX = c_set[l]
            break_class_X = False
            if _classX == 0:
                continue
            for j in range(l, len(c_set)):
                y, _classY = c_set[j]
                if x == y or _classX != _classY:
                    continue
                for k in range(len(c_set)):
                    z, _classZ = c_set[k]
                    # if _classZ == 0:
                    #     continue
                    if _classZ == _classX or x == z or y == z:
                        continue
                    positive_dist = np.linalg.norm(embedded_positions[x] - embedded_positions[y])
                    negative_dist = np.linalg.norm(embedded_positions[x] - embedded_positions[z])
                    if positive_dist + MARGIN < negative_dist:
                        correct += 1
                        class_accuracy[_classX] += 1
                    total += 1
                    class_counts[_classX] += 1

        print(class_counts)
        print(f"CLASS ACCURACY (Out of {total} triplets):")
        for class_value in class_accuracy:
            if class_value == 0 or class_counts[class_value] == 0:
                continue
            print(f"{class_value}: {class_accuracy[class_value] * 100 / class_counts[class_value]}")
        acc = correct * 100 / total
        a.append(acc)
        print(f"Ensemble {i} ~ Accuracy: {acc}")

    print(f"Average: {sum(a) / len(ensemble.ensemble)}")
    print("=" * 20)

Initialized Network with  512675 parameters
VAL results
Class Set of Size: 500
{0: 0, 1: 1357560, 2: 450450, 3: 1011126, 4: 730626, 5: 166023}
CLASS ACCURACY (Out of 3715785 triplets):
1: 71.84927369692684
2: 62.156732156732154
3: 66.25999133639131
4: 81.34969190803503
5: 87.32103383266174
Ensemble 0 ~ Accuracy: 71.71267982404795
Average: 71.71267982404795
